# Financial Expansion Readiness Project

This notebook trains the cleaner version of our model locally. It uses the new `EntireCompustat.csv` dataset, builds the target, trains a tuned logistic regression model, evaluates it, saves the artifacts for Streamlit, and lets us score a selected company.

## 1. Setup
We first locate the project folder and import the shared helper functions from `project_utils.py`. This keeps the notebook and Streamlit app consistent.

In [1]:

import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Try a few likely locations so the notebook works in Colab or locally.
PROJECT_CANDIDATES = [
    Path('.'),
    Path('/content'),
]

PROJECT_DIR = None
for candidate in PROJECT_CANDIDATES:
    if (candidate / 'project_utils.py').exists():
        PROJECT_DIR = str(candidate.resolve())
        break

if PROJECT_DIR is None:
    PROJECT_DIR = os.getcwd()

if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

from project_utils import (
    FEATURE_COLS,
    DISPLAY_COLS,
    TARGET_DESCRIPTION,
    load_wrds_data,
    build_features_and_target,
    prepare_model_data,
    split_train_val_test_by_year,
    train_and_tune_logistic,
    evaluate_model,
    get_model_coefficients,
    get_latest_company_rows,
    get_peer_medians,
    get_same_sic_peer_median,
    recommend_from_probability,
    save_artifacts,
)

DATA_PATHS = [
    os.path.join(PROJECT_DIR, 'data', 'EntireCompustat.csv'),
    os.path.join(PROJECT_DIR, 'EntireCompustat.csv'),
    '/content/EntireCompustat.csv',
]

data_path = None
for p in DATA_PATHS:
    if os.path.exists(p):
        data_path = p
        break

print("Project directory:", PROJECT_DIR)
print("Data path:", data_path)
print("Model features:", FEATURE_COLS)
print("Target rule:", TARGET_DESCRIPTION)


Project directory: /Users/visheshgoyal/Downloads/finance_project_final 
Data path: /Users/visheshgoyal/Downloads/finance_project_final /data/EntireCompustat.csv
Model features: ['revenue_growth', 'roa', 'debt_ratio', 'cfo_assets']
Target rule: Target = 1 when next-year revenue growth beats either the industry median or 5%, next-year ROA is above 2%, next-year CFO/Assets is positive, and next-year Debt/Assets is below 0.60.


**Interpretation:** if the data path prints correctly, the notebook found the new dataset and is ready to continue. The model features printed here are the only ratios used inside the regression.

## 2. Load the WRDS dataset
Here we load the raw Compustat file and inspect the shape, columns, and a few first rows.

In [2]:

raw_df = load_wrds_data(data_path)
print("Rows, columns:", raw_df.shape)
print("\nColumns:")
print(list(raw_df.columns))
raw_df.head()


Rows, columns: (526622, 24)

Columns:
['costat', 'curcd', 'datafmt', 'indfmt', 'consol', 'tic', 'fyear', 'gvkey', 'conm', 'sic', 'act', 'at', 'dlc', 'dltt', 'invt', 'lct', 'rect', 'ni', 'oiadp', 'sale', 'xint', 'oancf', 'sich', 'sic_final']


,costat,curcd,datafmt,indfmt,consol,tic,fyear,gvkey,conm,sic,...,invt,lct,rect,ni,oiadp,sale,xint,oancf,sich,sic_final
0,I,USD,STD,INDL,C,AE.2,1961,1000,A & E PLASTIK PAK INC,3089,...,NaN,NaN,NaN,NaN,NaN,0.900,NaN,NaN,NaN,3089.0
1,I,USD,STD,INDL,C,AE.2,1962,1000,A & E PLASTIK PAK INC,3089,...,NaN,NaN,NaN,NaN,NaN,1.600,0.010,NaN,NaN,3089.0
2,I,USD,STD,INDL,C,AE.2,1963,1000,A & E PLASTIK PAK INC,3089,...,0.133,0.322,0.250,0.003,0.000,1.457,0.020,NaN,NaN,3089.0
3,I,USD,STD,INDL,C,AE.2,1964,1000,A & E PLASTIK PAK INC,3089,...,0.204,0.267,0.230,0.052,0.074,2.032,0.033,NaN,NaN,3089.0
4,I,USD,STD,INDL,C,AE.2,1965,1000,A & E PLASTIK PAK INC,3089,...,0.378,0.623,0.291,-0.197,-0.242,1.688,0.062,NaN,NaN,3089.0


**Interpretation:** this tells us whether the file structure looks correct. We expect one row per firm-year, with identifiers such as `gvkey`, `tic`, `fyear`, and accounting fields such as `sale`, `at`, `ni`, `dlc`, `dltt`, and `oancf`.

## 3. Build ratios and the cleaner target
We now create the current-year ratios, next-year outcome variables, and the target variable.

### Regression ratios used
- `revenue_growth`
- `roa`
- `debt_ratio`
- `cfo_assets`

### Cleaner target
Target = 1 if next year the company:
- beats either its industry median revenue growth or 5%
- has ROA above 2%
- has positive CFO/Assets
- keeps Debt/Assets below 0.60

In [3]:

full_df = build_features_and_target(raw_df)
full_df[['tic', 'fyear', 'revenue_growth', 'roa', 'debt_ratio', 'cfo_assets', 'target']].head(10)


,tic,fyear,revenue_growth,roa,debt_ratio,cfo_assets,target
0,AE.2,1961,NaN,NaN,NaN,NaN,NaN
1,AE.2,1962,0.777778,NaN,NaN,NaN,NaN
2,AE.2,1963,-0.089375,NaN,NaN,NaN,NaN
3,AE.2,1964,0.394647,0.036723,0.430791,NaN,NaN
4,AE.2,1965,-0.169291,-0.085281,0.629437,NaN,NaN
5,AE.2,1966,1.388626,0.067490,0.507407,NaN,NaN
6,AE.2,1967,-0.108631,-0.036645,0.562296,NaN,NaN
7,AE.2,1968,1.058987,0.078183,0.259541,NaN,NaN
8,AE.2,1969,4.052973,0.061507,0.304542,NaN,NaN
9,AE.2,1970,0.212425,0.016682,0.397459,NaN,NaN


**Interpretation:** this preview lets us confirm that the engineered ratios and target were created correctly. Missing values are normal at this stage because lagged and next-year variables require more than one year of data.

## 4. Clean the modeling sample
We keep the sample cleaner by:
- starting at 2010
- removing rows with non-positive sales or assets
- removing duplicate firm-year rows
- requiring enough non-missing model ratios to be usable

In [4]:

model_df = prepare_model_data(full_df, start_year=2010)

print("Modeling rows:", len(model_df))
print("Years covered:", int(model_df['fyear'].min()), "to", int(model_df['fyear'].max()))
print("Positive target rate:", round(model_df['target'].mean(), 4))

model_df[FEATURE_COLS].describe().T[['count', 'mean', '50%']]


Modeling rows: 87120
Years covered: 2010 to 2024
Positive target rate: 0.2461


,count,mean,50%
revenue_growth,81280.0,1.268414,0.063014
roa,87120.0,-0.438406,0.011667
debt_ratio,87120.0,0.517689,0.213122
cfo_assets,87053.0,-0.097187,0.045822


**Interpretation:** the target rate tells us how many observations are labeled as financially expansion-ready. The ratio summary helps us check whether the cleaned sample still looks economically reasonable.

## 5. Time-based split
We use a time-based split to keep the evaluation realistic.
- Train: through 2020
- Validation: 2021 to 2022
- Test: 2023 onward

In [5]:

train_df, val_df, test_df = split_train_val_test_by_year(model_df, train_end_year=2020, val_end_year=2022)

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))
print("\nPositive rates:")
print("Train:", round(train_df['target'].mean(), 4))
print("Validation:", round(val_df['target'].mean(), 4))
print("Test:", round(test_df['target'].mean(), 4))


Train rows: 65285
Validation rows: 11914
Test rows: 9921

Positive rates:
Train: 0.2495
Validation: 0.2332
Test: 0.2391


**Interpretation:** the positive rate should not shift too wildly across the three periods. A time-based split is also easier to defend to a professor than a random split for financial data.

## 6. Train and tune the logistic regression model
We keep the model simple but tune two things:
- the regularization strength `C`
- the probability threshold used for the final classification

We do **not** force the default 0.50 threshold if the validation period suggests a better cutoff.

In [6]:

model, val_metrics = train_and_tune_logistic(train_df, val_df)

print("Selected C:", val_metrics['C'])
print("Selected threshold:", val_metrics['threshold'])
print("Validation accuracy:", round(val_metrics['val_accuracy'], 4))
print("Validation F1:", round(val_metrics['val_f1'], 4))
print("Validation ROC-AUC:", round(val_metrics['val_auc'], 4))


Selected C: 1.0
Selected threshold: 0.52
Validation accuracy: 0.7437
Validation F1: 0.5404
Validation ROC-AUC: 0.778


**Interpretation:** this section shows the tuned settings chosen from the validation period. The threshold is especially important because it controls how conservative or aggressive the classifier becomes.

## 7. Evaluate on the held-out test period

In [7]:

test_metrics = evaluate_model(model, test_df, threshold=val_metrics['threshold'])

for k, v in test_metrics.items():
    if k == 'confusion_matrix':
        continue
    print(f"{k}: {round(v, 4)}")

print("\nConfusion matrix:")
print(np.array(test_metrics['confusion_matrix']))


threshold: 0.52
accuracy: 0.7354
f1: 0.5604
precision: 0.4649
recall: 0.7053
balanced_accuracy: 0.7251
auc: 0.7927

Confusion matrix:
[[5623 1926]
 [ 699 1673]]


**Interpretation:**
- **accuracy** shows the overall share of correct classifications
- **F1** balances precision and recall
- **balanced accuracy** is useful when the classes are not perfectly balanced
- **AUC** measures ranking quality independent of the threshold

The confusion matrix tells us how many actual 0s and 1s were correctly or incorrectly classified.

## 8. Look at the coefficients
A logistic regression stays useful for class because we can still interpret the coefficient signs.

In [8]:

coef_df = get_model_coefficients(model)
coef_df


,feature,coefficient,abs_coefficient
1,roa,2.485636,2.485636
3,cfo_assets,2.073735,2.073735
2,debt_ratio,-0.437424,0.437424
0,revenue_growth,0.017330,0.017330


**Interpretation:** a positive coefficient means the ratio pushes the model more toward the expansion-ready class. A negative coefficient means the ratio pushes it away. Larger absolute values matter more inside the model.

## 9. Save the artifacts for Streamlit
This writes the fitted model and the supporting CSV/JSON files into the `artifacts/` folder so the Streamlit app can stay lightweight.

In [9]:

latest_df = get_latest_company_rows(full_df)
peer_df = get_peer_medians(latest_df)

metrics = {
    'target_description': TARGET_DESCRIPTION,
    'model_features': FEATURE_COLS,
    'display_features': DISPLAY_COLS,
    'start_year': 2010,
    'train_end_year': 2020,
    'val_end_year': 2022,
    'n_train_rows': int(len(train_df)),
    'n_val_rows': int(len(val_df)),
    'n_test_rows': int(len(test_df)),
    'positive_rate_train': float(train_df['target'].mean()),
    'positive_rate_val': float(val_df['target'].mean()),
    'positive_rate_test': float(test_df['target'].mean()),
    'selected_C': float(val_metrics['C']),
    'selected_threshold': float(val_metrics['threshold']),
    'validation_accuracy': float(val_metrics['val_accuracy']),
    'validation_f1': float(val_metrics['val_f1']),
    'validation_balanced_accuracy': float(val_metrics['val_balanced_accuracy']),
    'validation_auc': float(val_metrics['val_auc']),
    'test_accuracy': float(test_metrics['accuracy']),
    'test_f1': float(test_metrics['f1']),
    'test_precision': float(test_metrics['precision']),
    'test_recall': float(test_metrics['recall']),
    'test_balanced_accuracy': float(test_metrics['balanced_accuracy']),
    'test_auc': float(test_metrics['auc']),
    'confusion_matrix': test_metrics['confusion_matrix'],
}

artifact_dir = save_artifacts(PROJECT_DIR, model, latest_df, peer_df, metrics, coef_df)
print("Artifacts saved to:", artifact_dir)


Artifacts saved to: /Users/visheshgoyal/Downloads/finance_project_final /artifacts


**Interpretation:** if the artifact folder prints correctly, the app can load the saved model without retraining inside Streamlit.

## 10. Score a selected company inside the notebook
This gives us a notebook-based demo in case we want to present without opening Streamlit.

In [10]:

sorted_tickers = sorted(latest_df['tic'].dropna().unique().tolist())
print("Number of latest company records:", len(sorted_tickers))
print("Example tickers:", sorted_tickers[:20])


Number of latest company records: 32223
Example tickers: ['0015B', '0030B', '0032A', '0033A', '0038A', '0039A', '0040A', '0041A', '0043A', '0044A', '0046A', '0047A', '0049A', '0050A', '0051A', '0052A', '0053A', '0054A', '0055A', '0055B']


In [11]:

# Change this ticker manually if you want.
selected_ticker = sorted_tickers[0]
selected_ticker


'0015B'

In [12]:
company_row = latest_df[latest_df['tic'] == selected_ticker].iloc[0]
peer_row = get_same_sic_peer_median(latest_df, company_row)

probability = float(model.predict_proba(pd.DataFrame([company_row])[FEATURE_COLS])[0, 1])
recommendation = recommend_from_probability(probability)

print("Company:", company_row['conm'])
print("Ticker:", company_row['tic'])
print("Latest fiscal year:", int(company_row['fyear']))
print("SIC used for comparison:", int(company_row['sic_final']) if pd.notna(company_row['sic_final']) else "Missing")
print("Same-SIC peer count used:", int(peer_row.get('peer_count', 0)) if peer_row is not None else 0)
print("Predicted probability:", round(probability, 4))
print("Recommendation:", recommendation)

comparison_df = pd.DataFrame({
    'Ratio': DISPLAY_COLS,
    'Company': [company_row.get(col, np.nan) for col in DISPLAY_COLS],
    'Same-SIC Peer Median': [peer_row.get(col, np.nan) if hasattr(peer_row, 'get') else np.nan for col in DISPLAY_COLS],
})
comparison_df['Difference'] = comparison_df['Company'] - comparison_df['Same-SIC Peer Median']
comparison_df


Company: BURLINGTON COAT FACTORY INVS
Ticker: 0015B
Latest fiscal year: 2013
SIC used for comparison: 5651
Same-SIC peer count used: 31
Predicted probability: 0.5307
Recommendation: Proceed Carefully


,Ratio,Company,Same-SIC Peer Median,Difference
0,revenue_growth,0.071176,0.039490,0.031686
1,roa,0.016737,0.011051,0.005686
2,debt_ratio,0.498114,0.386582,0.111532
3,cfo_assets,0.120095,0.021811,0.098284
4,current_ratio,1.182906,1.582725,-0.399818
5,profit_margin,0.009805,0.006542,0.003264


**Interpretation:** this section is our notebook demo. It shows the selected company’s predicted probability, recommendation, and ratio-by-ratio comparison against the median of companies with the same SIC code.

## 11. Optional widget-based selector for Colab
If widgets load properly, we can use the dropdown below. If not, the manual ticker cell above is enough.

In [13]:

try:
    import ipywidgets as widgets
    from IPython.display import display

    dropdown = widgets.Dropdown(
        options=sorted_tickers,
        value=selected_ticker,
        description='Ticker:',
        layout=widgets.Layout(width='300px')
    )
    display(dropdown)
except Exception as e:
    print("Widget could not be loaded:", e)


Dropdown(description='Ticker:', layout=Layout(width='300px'), options=('0015B', '0030B', '0032A', '0033A', '00…